In [1]:
# ===================== HEART ATTACK — ADVANCED FINAL MODEL =====================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN

from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

np.random.seed(42)

df = pd.read_csv("heartattack.csv")

# Convert to binary
y = (df["target"] != 0).astype(int)
X = pd.get_dummies(df.drop(columns=["target"]), drop_first=True)

# Impute
X = SimpleImputer(strategy="median").fit_transform(X)

# ============================================================
# BALANCING METHODS TESTED
# ============================================================
variants = {
    "SMOTE": SMOTE(random_state=42),
    "ADASYN": ADASYN(random_state=42),
    "TOMEK": SMOTETomek(random_state=42),
    "SMOTE+ENN": SMOTEENN(random_state=42)
}

best_model, best_score = None, -1
best_name = ""
best_metrics = {}

print("\n=========== HEART ATTACK — FINAL MODEL SELECTION ===========\n")

for method, resamp in variants.items():

    Xr, yr = resamp.fit_resample(X, y)

    scaler = StandardScaler()
    Xr = scaler.fit_transform(Xr)

    skb = SelectKBest(mutual_info_classif, k=min(30, Xr.shape[1]))
    X_filtered = skb.fit_transform(Xr, yr)

    pca = PCA(n_components=min(18, X_filtered.shape[1]))
    X_final = pca.fit_transform(X_filtered)

    Xtr,Xte,ytr,yte = train_test_split(
        X_final, yr, test_size=0.20, stratify=yr, random_state=42
    )

    # ==================== MODELS SAME UNCHANGED ====================
    candidates = {
        "XGBoost": XGBClassifier(
            n_estimators=700, max_depth=7, learning_rate=0.03,
            subsample=0.85, colsample_bytree=0.85, eval_metric="logloss"
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=700, learning_rate=0.03,
            num_leaves=40, max_depth=-1, class_weight="balanced"
        ),
        "CatBoost": CatBoostClassifier(
            depth=7, learning_rate=0.04, iterations=800,
            verbose=0, loss_function="Logloss"
        ),
        "BalancedRF": BalancedRandomForestClassifier(
            n_estimators=600, max_depth=None, random_state=42
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=400, learning_rate=0.03, max_depth=3
        )
    }

    print(f"\n=== {method} ===")
    for name, model in candidates.items():
        model.fit(Xtr, ytr)
        prob = model.predict_proba(Xte)[:,1]
        pred = (prob > 0.5).astype(int)

        acc = accuracy_score(yte,pred)
        f1  = f1_score(yte,pred)
        auc = roc_auc_score(yte,prob)
        mAP = average_precision_score(yte,prob)   # ← added

        print(f"{name:<15}| ACC={acc:.4f} | F1={f1:.4f} | AUC={auc:.4f} | mAP={mAP:.4f}")

        score = (auc*0.55 + f1*0.45)

        if score > best_score:
            best_score = score
            best_name = f"{name} [{method}]"
            best_metrics = {"ACC":acc,"F1":f1,"AUC":auc,"mAP":mAP}


# ============================================================
# FINAL RESULT (NO SAVING)
# ============================================================
print("\n========================================================")
print(f"🔥 BEST MODEL → {best_name}")
print(f"📌 ACC={best_metrics['ACC']:.4f} | F1={best_metrics['F1']:.4f} | AUC={best_metrics['AUC']:.4f}")
print(f"💥 mAP={best_metrics['mAP']:.4f}")   # ← REQUIRED OUTPUT
print("========================================================\n")



=========== HEART ATTACK — FINAL MODEL SELECTION ===========


=== SMOTE ===
XGBoost        | ACC=0.8317 | F1=0.8284 | AUC=0.8642 | mAP=0.9029
[LightGBM] [Info] Number of positive: 830, number of negative: 830
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000526 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2805
[LightGBM] [Info] Number of data points in the train set: 1660, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM       | ACC=0.8365 | F1=0.8333 | AUC=0.8612 | mAP=0.8932
CatBoost       | ACC=0.8245 | F1=0.8206 | AUC=0.8638 | mAP=0.8984
BalancedRF     | ACC=0.8221 | F1=0.8213 | AUC=0.8504 | mAP=0.8923
GradientBoosting| ACC=0.7933 | F1=0.7892 | AUC=0.8557 | mAP=0.8709

=== ADASYN ===
XGBoost        | ACC=0.8015 | F1=0.8106 | AUC=0.8650 | mAP=0.9120
[LightGBM] [Info] Number of positive: 830, number of negative: 758
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000147 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2805
[LightGBM] [Info] Number of data points in the train set: 1588, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM       | ACC=0.8342 | F1=0.8358 | AUC=0.8667 | mAP=0.9096
CatBoost       | ACC=0.8065 | F1=0.8136 | AUC=0.8670 | mAP=0.9108
BalancedRF     | ACC=0.8065 | F1=0.8108 | AUC=0.8526 | mAP=0.9071
GradientBoosting| ACC=0.7940 | F1=0.7990 | AUC=0.8672 | mAP=0.9003

=== TOMEK ===
XGBoost        | ACC=0.8148 | F1=0.8175 | AUC=0.8786 | mAP=0.9127
[LightGBM] [Info] Number of positive: 810, number of negative: 809
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001834 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2805
[LightGBM] [Info] Number of data points in the train set: 1619, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM       | ACC=0.8247 | F1=0.8256 | AUC=0.8784 | mAP=0.9121
CatBoost       | ACC=0.8346 | F1=0.8321 | AUC=0.8817 | mAP=0.9167
BalancedRF     | ACC=0.8247 | F1=0.8264 | AUC=0.8743 | mAP=0.9133
GradientBoosting| ACC=0.7778 | F1=0.7716 | AUC=0.8710 | mAP=0.8924

=== SMOTE+ENN ===
XGBoost        | ACC=0.9575 | F1=0.9677 | AUC=0.9898 | mAP=0.9953
[LightGBM] [Info] Number of positive: 555, number of negative: 292
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000237 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2779
[LightGBM] [Info] Number of data points in the train set: 847, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Wa

C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


CatBoost       | ACC=0.9670 | F1=0.9747 | AUC=0.9921 | mAP=0.9964
BalancedRF     | ACC=0.9623 | F1=0.9706 | AUC=0.9912 | mAP=0.9958
GradientBoosting| ACC=0.9198 | F1=0.9399 | AUC=0.9815 | mAP=0.9913

🔥 BEST MODEL → LightGBM [SMOTE+ENN]
📌 ACC=0.9670 | F1=0.9749 | AUC=0.9933
💥 mAP=0.9969

